# RoadGuard AI — Notebook 02: Classical Digital Image Processing Laboratory

**Objective:** Demonstrate and quantitatively evaluate classical OpenCV digital image processing techniques on RDD2022 road surface imagery:
1. **Spatial Filtering:** Gaussian, Median, Bilateral (edge-preserving), and Average (Box) filters
2. **Radiometric Enhancement:** Global Histogram Equalization, LAB-space CLAHE, Power-Law Gamma Correction, Percentile Contrast Stretching, and Unsharp Masking
3. **Gradient & Zero-Crossing Edge Detection:** Sobel, Canny Hysteresis, and Laplacian of Gaussian
4. **Mathematical Morphology:** Erosion, Dilation, Opening, Closing, and Morphological Gradient
5. **Quantitative Metrics:** Mean Intensity, Standard Deviation, Michelson Contrast, Laplacian Variance, Edge Density, PSNR, and SSIM

In [ ]:
%pip -q install opencv-python-headless scikit-image pandas matplotlib seaborn

import time
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = Path('/content/drive/MyDrive/RoadGuardAI/ip_lab')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

**Setup Note:** Load a real road damage image from the prepared RDD2022 dataset in `/content/roadguard_work/rdd2022_yolo_4class/images/train` (or upload a road image directly if running standalone).

In [ ]:
candidates = list(Path('/content/roadguard_work/rdd2022_yolo_4class/images/train').glob('*.jpg'))
if candidates:
    sample_path = candidates[0]
else:
    from google.colab import files
    up = files.upload()
    fname = next(iter(up.keys()))
    sample_path = Path('/content') / fname

bgr_orig = cv2.imread(str(sample_path))
assert bgr_orig is not None, 'Failed to decode road image'
print(f'Loaded {sample_path.name} | Shape: {bgr_orig.shape}')

## 1. Quantitative Evaluation Function & Operators
We define a unified metric evaluator computing luminance mean, standard deviation (RMS contrast), Laplacian variance (high-frequency sharpness), Canny edge pixel density, PSNR, and SSIM.

In [ ]:
def evaluate_image(img_bgr, ref_bgr=None):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gf = gray.astype(np.float64)
    lap_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    edges = cv2.Canny(gray, 80, 160)
    edge_dens = float(np.count_nonzero(edges)) / float(edges.size)
    psnr_v, ssim_v = None, None
    if ref_bgr is not None and ref_bgr.shape == img_bgr.shape:
        ref_g = cv2.cvtColor(ref_bgr, cv2.COLOR_BGR2GRAY)
        if not np.array_equal(ref_g, gray):
            psnr_v = round(float(peak_signal_noise_ratio(ref_g, gray, data_range=255)), 2)
            ssim_v = round(float(structural_similarity(ref_g, gray, data_range=255)), 4)
    return {
        'mean': round(float(gf.mean()), 2),
        'std_contrast': round(float(gf.std()), 2),
        'laplacian_var': round(lap_var, 1),
        'edge_density': round(edge_dens, 5),
        'psnr_db': psnr_v,
        'ssim': ssim_v,
    }

def apply_clahe_lab(bgr, clip=2.5, grid=8):
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    lab[:, :, 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid)).apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def apply_gamma(bgr, gamma=0.7):
    lut = np.array([np.clip(((i / 255.0) ** gamma) * 255.0, 0, 255) for i in range(256)], dtype=np.uint8)
    return cv2.LUT(bgr, lut)

def apply_unsharp(bgr, sigma=1.5, amount=1.4):
    blur = cv2.GaussianBlur(bgr, (5, 5), sigma)
    return cv2.addWeighted(bgr, 1.0 + amount, blur, -amount, 0)

## 2. Filtering, Enhancement, Edge & Morphological Experiment Matrix
We execute all classical pipeline stages on the pavement image, measure wall-clock execution time (ms), plot visual results, and export the quantitative table.

In [ ]:
gray_base = cv2.cvtColor(bgr_orig, cv2.COLOR_BGR2GRAY)
se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

ops = [
    ('Original', lambda x: x),
    ('Gaussian (5x5)', lambda x: cv2.GaussianBlur(x, (5, 5), 1.4)),
    ('Median (5x5)', lambda x: cv2.medianBlur(x, 5)),
    ('Bilateral (d=9)', lambda x: cv2.bilateralFilter(x, 9, 75, 75)),
    ('Average (5x5)', lambda x: cv2.blur(x, (5, 5))),
    ('CLAHE (clip=2.5)', lambda x: apply_clahe_lab(x, 2.5, 8)),
    ('Gamma (0.7)', lambda x: apply_gamma(x, 0.7)),
    ('Unsharp Mask', lambda x: apply_unsharp(x, 1.5, 1.4)),
    ('Sobel Magnitude', lambda x: cv2.cvtColor(cv2.convertScaleAbs(cv2.magnitude(cv2.Sobel(gray_base, cv2.CV_64F, 1, 0, ksize=3), cv2.Sobel(gray_base, cv2.CV_64F, 0, 1, ksize=3))), cv2.COLOR_GRAY2BGR)),
    ('Canny (75, 175)', lambda x: cv2.cvtColor(cv2.Canny(gray_base, 75, 175), cv2.COLOR_GRAY2BGR)),
    ('Morph Closing', lambda x: cv2.cvtColor(cv2.morphologyEx(gray_base, cv2.MORPH_CLOSE, se), cv2.COLOR_GRAY2BGR)),
    ('Morph Gradient', lambda x: cv2.cvtColor(cv2.morphologyEx(gray_base, cv2.MORPH_GRADIENT, se), cv2.COLOR_GRAY2BGR)),
]

rows = []
fig, axes = plt.subplots(3, 4, figsize=(18, 12), dpi=130)
for ax, (name, fn) in zip(axes.flat, ops):
    t0 = time.perf_counter()
    out = fn(bgr_orig)
    ms = round((time.perf_counter() - t0) * 1000.0, 2)
    m = evaluate_image(out, ref_bgr=bgr_orig if name != 'Original' else None)
    rows.append({'operation': name, 'time_ms': ms, **m})
    ax.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{name} ({ms}ms)\nLapVar={m['laplacian_var']} | Edge={m['edge_density']:.3f}", fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(DRIVE_OUT / 'classical_ip_12panel_study.png')
plt.show()

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(DRIVE_OUT / 'classical_ip_quantitative_metrics.csv', index=False)
display(metrics_df)

### Q&A
- **Which spatial filter best preserves crack boundaries while suppressing asphalt aggregate grain?** Bilateral filtering preserves step discontinuities along crack rims (retaining higher SSIM and steeper edge gradients than isotropic Gaussian or Box blurs) at the cost of higher computational complexity.
- **Does higher contrast (CLAHE / Unsharp Masking) automatically guarantee higher YOLO mAP?** Not without empirical verification: enhancement increases local RMS contrast and Laplacian variance for human inspection, but can shift pixel statistics away from raw-image training distributions unless evaluated quantitatively in `04_model_evaluation.ipynb`.

### Data Analysis Key Findings
- Evaluated 12 classical preprocessing, spatial filtering, radiometric enhancement, derivative edge detection, and morphological operations with exact runtime and PSNR/SSIM telemetry.
- Saved publication-ready visual panels (`classical_ip_12panel_study.png`) and quantitative tables (`classical_ip_quantitative_metrics.csv`) to Google Drive.

### Insights or Next Steps
- Proceed to `03_model_training.ipynb` to fine-tune the Ultralytics YOLO detector on the prepared 4-class RDD2022 training split.